In [1]:
pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from io import StringIO
import warnings
warnings.filterwarnings('ignore')


In [3]:
df = pd.read_csv("data set/HRDataset_v14.csv")
print("✅ Data loaded:", df.shape)

✅ Data loaded: (311, 36)


In [4]:
df.head()

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,"Adinolfi, Wilson K",10026,0,0,1,1,5,4,0,62506,...,Michael Albert,22.0,LinkedIn,Exceeds,4.60,5,0,1/17/2019,0,1
1,"Ait Sidi, Karthikeyan",10084,1,1,1,5,3,3,0,104437,...,Simon Roup,4.0,Indeed,Fully Meets,4.96,3,6,2/24/2016,0,17
2,"Akinkuolie, Sarah",10196,1,1,0,5,5,3,0,64955,...,Kissy Sullivan,20.0,LinkedIn,Fully Meets,3.02,3,0,5/15/2012,0,3
3,"Alagbe,Trina",10088,1,1,0,1,5,3,0,64991,...,Elijiah Gray,16.0,Indeed,Fully Meets,4.84,5,0,1/3/2019,0,15
4,"Anderson, Carol",10069,0,2,0,5,5,3,0,50825,...,Webster Butler,39.0,Google Search,Fully Meets,5.00,4,0,2/1/2016,0,2


In [5]:
df.columns = df.columns.str.strip()
df['Department'] = df['Department'].str.strip()
df['EmploymentStatus'] = df['EmploymentStatus'].str.strip()
df['RecruitmentSource'] = df['RecruitmentSource'].str.strip()
df['TermReason'] = df['TermReason'].str.strip()

In [6]:
# Parse dates
df['DateofHire'] = pd.to_datetime(df['DateofHire'], errors='coerce')
df['DateofTermination'] = pd.to_datetime(df['DateofTermination'], errors='coerce')

In [7]:
# Extract hire year and month
df['HireYear'] = df['DateofHire'].dt.year
df['HireMonth'] = df['DateofHire'].dt.month
df['HireYearMonth'] = df['DateofHire'].dt.to_period('M')

print("✅ Cleaned. Hire years range:", df['HireYear'].min(), "to", df['HireYear'].max())
print("Departments:", df['Department'].unique())

✅ Cleaned. Hire years range: 2006 to 2018
Departments: ['Production' 'IT/IS' 'Software Engineering' 'Admin Offices' 'Sales'
 'Executive Office']


In [8]:
## 3A. Hires per Year
hires_per_year = df.groupby('HireYear').size().reset_index(name='Hires')
print("\n📊 Hires per Year:\n", hires_per_year)

## 3B. Hires per Department
hires_per_dept = df.groupby('Department').size().sort_values(ascending=False).reset_index(name='Hires')
print("\n📊 Hires per Department:\n", hires_per_dept)

## 3C. Terminations per Department
term_df = df[df['Termd'] == 1]
term_per_dept = term_df.groupby('Department').size().sort_values(ascending=False).reset_index(name='Terminations')
print("\n📊 Terminations per Department:\n", term_per_dept)

## 3D. Recruitment Source breakdown
recruit_source = df['RecruitmentSource'].value_counts().reset_index()
recruit_source.columns = ['Source', 'Count']
print("\n📊 Recruitment Sources:\n", recruit_source)


📊 Hires per Year:
     HireYear  Hires
0       2006      1
1       2007      2
2       2008      3
3       2009      7
4       2010      9
5       2011     83
6       2012     45
7       2013     44
8       2014     60
9       2015     36
10      2016     14
11      2017      6
12      2018      1

📊 Hires per Department:
              Department  Hires
0            Production    209
1                 IT/IS     50
2                 Sales     31
3  Software Engineering     11
4         Admin Offices      9
5      Executive Office      1

📊 Terminations per Department:
              Department  Terminations
0            Production            83
1                 IT/IS            10
2                 Sales             5
3  Software Engineering             4
4         Admin Offices             2

📊 Recruitment Sources:
                     Source  Count
0                   Indeed     87
1                 LinkedIn     76
2            Google Search     49
3        Employee Referral     31
4

In [9]:
dept_year = df.groupby(['Department', 'HireYear']).size().reset_index(name='Hires')

forecast_results = []
future_years = [2019, 2020, 2021]

for dept in dept_year['Department'].unique():
    subset = dept_year[dept_year['Department'] == dept]
    if len(subset) < 2:
        continue
    X = subset['HireYear'].values.reshape(-1, 1)
    y = subset['Hires'].values
    model = LinearRegression()
    model.fit(X, y)
    for yr in future_years:
        pred = model.predict([[yr]])[0]
        forecast_results.append({
            'Department': dept,
            'Year': yr,
            'Predicted_Hires': max(0, round(pred, 1))
        })

forecast_df = pd.DataFrame(forecast_results)
print("\n🔮 Forecast (2019–2021):\n", forecast_df)
forecast_df.to_csv("forecast_output.csv", index=False)
print("✅ Forecast saved to forecast_output.csv")


🔮 Forecast (2019–2021):
               Department  Year  Predicted_Hires
0          Admin Offices  2019              2.4
1          Admin Offices  2020              2.6
2          Admin Offices  2021              2.7
3                  IT/IS  2019             12.7
4                  IT/IS  2020             13.8
5                  IT/IS  2021             15.0
6             Production  2019             21.2
7             Production  2020             21.5
8             Production  2021             21.8
9                  Sales  2019              5.5
10                 Sales  2020              5.8
11                 Sales  2021              6.0
12  Software Engineering  2019              0.0
13  Software Engineering  2020              0.0
14  Software Engineering  2021              0.0
✅ Forecast saved to forecast_output.csv
